# A sentiment analysis of SONA using VADER and roBERTa

In [1]:
!pip3 install --upgrade --force-reinstall stopwordsiso
!pip3 install --upgrade typing_extensions

  Using cached stopwordsiso-0.7.1-py3-none-any.whl.metadata (3.5 kB)
Using cached stopwordsiso-0.7.1-py3-none-any.whl (74 kB)
  Attempting uninstall: stopwordsiso
    Found existing installation: stopwordsiso 0.7.1
    Uninstalling stopwordsiso-0.7.1:
      Successfully uninstalled stopwordsiso-0.7.1


In [2]:
import pandas as pd
import numpy as np
import nltk
import numpy as np
import re
import altair as alt
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
import stopwordsiso as stopwords

from textblob import TextBlob
from textblob import Blobber
from textblob.sentiments import NaiveBayesAnalyzer
from nltk.sentiment.vader import SentimentIntensityAnalyzer

In [3]:
df= pd.read_csv('./csv/merged_real.csv')
df

,president,date,title,link,venue,session,speech,total_words
0,Manuel L. Quezon,"November 25, 1935",Message to the First Assembly on National Defense,https://www.officialgazette.gov.ph/1935/11/25/...,"Legislative Building, Manila","First National Assembly, First Session","Mr. Speaker, gentlemen of the National Assemb...",4341
1,Manuel L. Quezon,"June 16, 1936",On the Country’s Conditions and Problems,https://www.officialgazette.gov.ph/1936/06/16/...,"Legislative Building, Manila","First National Assembly, First Session","Mr. Speaker, Gentlemen of the National Assemb...",7250
2,Manuel L. Quezon,"October 18, 1937","Improvement of Philippine Conditions, Philippi...",https://www.officialgazette.gov.ph/1937/10/18/...,"Legislative Building, Manila","First National Assembly, Second Session","Mr. Speaker, Gentlemen of the National Assemb...",5774
3,Manuel L. Quezon,"January 24, 1938",Revision of the System of Taxation,https://www.officialgazette.gov.ph/1938/01/24/...,"Legislative Building, Manila","First National Assembly, Third Session",Gentlemen of the National Assembly: The state...,3212
4,Manuel L. Quezon,"January 24, 1939",The State of the Nation and Important Economic...,https://www.officialgazette.gov.ph/1939/01/24/...,"Legislative Building, Manila","Second National Assembly, First Session",Gentlemen of the National Assembly: I take pl...,4826
...,...,...,...,...,...,...,...,...
82,Rodrigo Roa Duterte,"July 26, 2021",Sixth State of the Nation Address,https://www.officialgazette.gov.ph/2021/07/26/...,"Batasang Pambansa, Quezon City","Eighteenth Congress, Third Session",Kindly sit down. By far this is the most bea...,14769
83,Ferdinand R. Marcos Jr.,"July 25, 2022",First State of the Nation Address,https://www.officialgazette.gov.ph/2021/07/26/...,"Batasang Pambansa, Quezon City","Nineteenth Congress, First Session",Vice President Sara Zimmerman Duterte; Form...,7990
84,Ferdinand R. Marcos Jr.,"July 24, 2023",Second State of the Nation Address,https://www.officialgazette.gov.ph/sf0clA,"Batasang Pambansa, Quezon City","Nineteenth Congress, Second Session",Thank you. Thank you very much. Allow me to g...,8227
85,Ferdinand R. Marcos Jr.,"July 22, 2024",Third State of the Nation Address,https://www.officialgazette.gov.ph/YdHHeA,"Batasang Pambansa, Quezon City","Nineteenth Congress, Third Session",Thank you. Allow me to greet the former Presi...,9177


## Clean data first

Remove Diosdado Macapagal speech in 1965 because it was not scraped.

In [4]:
df = df.drop(27)

## Run NLTK and TextBlob

Both lexicon-based language model.

In [5]:
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('brown')
nltk.download('averaged_perceptron_tagger')
nltk.download('conll2000')
nltk.download('movie_reviews')
nltk.download('vader_lexicon')

[nltk_data] Downloading package punkt to /Users/U6120756/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/U6120756/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package brown to /Users/U6120756/nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/U6120756/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package conll2000 to
[nltk_data]     /Users/U6120756/nltk_data...
[nltk_data]   Package conll2000 is already up-to-date!
[nltk_data] Downloading package movie_reviews to
[nltk_data]     /Users/U6120756/nltk_data...
[nltk_data]   Package movie_reviews is already up-to-date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/U6120756/nltk_data...
[nltk_data]   Package vader

True

In [6]:
sia = SentimentIntensityAnalyzer()

def get_scores(content):
    blob = TextBlob(content)
    sia_scores = sia.polarity_scores(content)

    return pd.Series({
        'textblob': blob.sentiment.polarity,
        'nltk': sia_scores['compound'],
    })

df['speech'] = df['speech'].fillna('')
scores = df['speech'].apply(get_scores)
df = pd.concat([df, scores], axis=1)

def label_sentiment(score):
    if score >= 0.05:
        return 'positive'
    elif score <= -0.05:
        return 'negative'
    else:
        return 'neutral'

df['sentiment_label'] = df['nltk'].apply(label_sentiment)

In [7]:
df

,president,date,title,link,venue,session,speech,total_words,textblob,nltk,sentiment_label
0,Manuel L. Quezon,"November 25, 1935",Message to the First Assembly on National Defense,https://www.officialgazette.gov.ph/1935/11/25/...,"Legislative Building, Manila","First National Assembly, First Session","Mr. Speaker, gentlemen of the National Assemb...",4341,0.106794,0.9999,positive
1,Manuel L. Quezon,"June 16, 1936",On the Country’s Conditions and Problems,https://www.officialgazette.gov.ph/1936/06/16/...,"Legislative Building, Manila","First National Assembly, First Session","Mr. Speaker, Gentlemen of the National Assemb...",7250,0.110458,1.0000,positive
2,Manuel L. Quezon,"October 18, 1937","Improvement of Philippine Conditions, Philippi...",https://www.officialgazette.gov.ph/1937/10/18/...,"Legislative Building, Manila","First National Assembly, Second Session","Mr. Speaker, Gentlemen of the National Assemb...",5774,0.137388,1.0000,positive
3,Manuel L. Quezon,"January 24, 1938",Revision of the System of Taxation,https://www.officialgazette.gov.ph/1938/01/24/...,"Legislative Building, Manila","First National Assembly, Third Session",Gentlemen of the National Assembly: The state...,3212,0.060580,0.9995,positive
4,Manuel L. Quezon,"January 24, 1939",The State of the Nation and Important Economic...,https://www.officialgazette.gov.ph/1939/01/24/...,"Legislative Building, Manila","Second National Assembly, First Session",Gentlemen of the National Assembly: I take pl...,4826,0.121574,1.0000,positive
...,...,...,...,...,...,...,...,...,...,...,...
82,Rodrigo Roa Duterte,"July 26, 2021",Sixth State of the Nation Address,https://www.officialgazette.gov.ph/2021/07/26/...,"Batasang Pambansa, Quezon City","Eighteenth Congress, Third Session",Kindly sit down. By far this is the most bea...,14769,0.133088,1.0000,positive
83,Ferdinand R. Marcos Jr.,"July 25, 2022",First State of the Nation Address,https://www.officialgazette.gov.ph/2021/07/26/...,"Batasang Pambansa, Quezon City","Nineteenth Congress, First Session",Vice President Sara Zimmerman Duterte; Form...,7990,0.119099,1.0000,positive
84,Ferdinand R. Marcos Jr.,"July 24, 2023",Second State of the Nation Address,https://www.officialgazette.gov.ph/sf0clA,"Batasang Pambansa, Quezon City","Nineteenth Congress, Second Session",Thank you. Thank you very much. Allow me to g...,8227,0.139571,1.0000,positive
85,Ferdinand R. Marcos Jr.,"July 22, 2024",Third State of the Nation Address,https://www.officialgazette.gov.ph/YdHHeA,"Batasang Pambansa, Quezon City","Nineteenth Congress, Third Session",Thank you. Allow me to greet the former Presi...,9177,0.139383,1.0000,positive


## Double checking the data

Notice that all negative sentiment was from Aquino's SONA. Aquino spoke largely or totally in Filipino on his SONA. Filipino is not captured by either TextBlob or VADER.

In [8]:
df.sentiment_label.value_counts()

positive    78
negative     8
Name: sentiment_label, dtype: int64

In [9]:
df[df.sentiment_label=='negative']

,president,date,title,link,venue,session,speech,total_words,textblob,nltk,sentiment_label
71,Benigno S. Aquino III,"July 26, 2010",State of the Nation Address,https://www.officialgazette.gov.ph/2010/07/26/...,"Batasang Pambansa, Quezon City","Fifteenth Congress, First Session",Maraming salamat po. Maupo po tayong lahat. S...,3828,-0.023173,-0.9998,negative
72,Benigno S. Aquino III,"July 25, 2011",Second State of the Nation Address,https://www.officialgazette.gov.ph/2011/07/25/...,"Batasang Pambansa, Quezon City","Fifteenth Congress, Second Session",Senate President Juan Ponce Enrile; Speaker F...,6217,0.047288,-1.0000,negative
73,Benigno S. Aquino III,"July 23, 2012",Third State of the Nation Address,https://www.officialgazette.gov.ph/2012/07/23/...,"Batasang Pambansa, Quezon City","Fifteenth Congress, Third Session",Maraming salamat po. Maupo ho tayong lahat. S...,9515,0.031271,-1.0000,negative
74,Benigno S. Aquino III,"July 22, 2013",Fourth State of the Nation Address,https://www.officialgazette.gov.ph/2013/07/22/...,"Batasang Pambansa, Quezon City","Sixteenth Congress, First Session",Marami pong salamat. Maupo ho tayong lahat. B...,12890,0.063955,-1.0000,negative
75,Benigno S. Aquino III,"July 28, 2014",Fifth State of the Nation Address,https://www.officialgazette.gov.ph/2014/07/28/...,"Batasang Pambansa, Quezon City","Sixteenth Congress, Second Session",Bise Presidente Jejomar Binay; dating Pangulo...,10234,0.061411,-1.0000,negative
76,Benigno S. Aquino III,"July 27, 2015",Sixth State of the Nation Address,https://www.officialgazette.gov.ph/2015/07/27/...,"Batasang Pambansa, Quezon City","Sixteenth Congress, Third Session",Maraming salamat po. Maupo ho tayo lahat. Bag...,14943,0.029932,-1.0000,negative
78,Rodrigo Roa Duterte,"July 24, 2017",Second State of the Nation Address,https://www.officialgazette.gov.ph/2017/07/24/...,"Batasang Pambansa, Quezon City","Seventeenth Congress, Second Session",Kindly sit down. Thank you for your courtes...,12712,0.055290,-0.9980,negative
86,Ferdinand R. Marcos Jr.,"July 28, 2025",Fourth State of the Nation Address,https://www.officialgazette.gov.ph/qF58Tj,"Batasang Pambansa, Quezon City","Twentieth Congress, First Session",Thank you. Thank you very much. Allow me to g...,7389,0.126499,-0.9905,negative


## RoBERTa model

In [10]:
# from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

# # 1. Load tokenizer, model, and pipeline
# model_name = "dost-asti/RoBERTa-tl-sentiment-analysis"
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModelForSequenceClassification.from_pretrained(model_name)

# sentiment_pipeline = pipeline(
#     "text-classification",
#     model=model,
#     tokenizer=tokenizer,
#     top_k=None
# )

# # 2. Confirmed label mapping
# label_map = {
#     'label_0': 'negative',
#     'label_1': 'positive',
#     'label_2': 'neutral'
# }

# # 3. Chunking function — leave room for special tokens (usually 2: <s> and </s>)
# def chunk_text(text, tokenizer, max_length=510, stride=50):
#     tokens = tokenizer.encode(text, add_special_tokens=False)
#     chunks = []
#     for i in range(0, len(tokens), max_length - stride):
#         chunk_tokens = tokens[i:i + max_length]
#         chunk_text = tokenizer.decode(chunk_tokens)
#         chunks.append(chunk_text)
#         if i + max_length >= len(tokens):
#             break
#     return chunks

# # 4. Sentiment scoring function — truncation=True as a safety net
# def get_sentiment(text, tokenizer=tokenizer, pipe=sentiment_pipeline):
#     if not isinstance(text, str) or not text.strip():
#         return pd.Series({'label': 'neutral', 'pos_score': 0, 'neg_score': 0, 'neutral_score': 0})

#     chunks = chunk_text(text, tokenizer)
#     all_scores = {'positive': [], 'negative': [], 'neutral': []}

#     for chunk in chunks:
#         result = pipe(chunk, truncation=True, max_length=510)[0]
#         for item in result:
#             raw_label = item['label'].lower()
#             mapped_label = label_map.get(raw_label, raw_label)
#             if mapped_label in all_scores:
#                 all_scores[mapped_label].append(item['score'])

#     avg_pos = sum(all_scores['positive']) / len(all_scores['positive']) if all_scores['positive'] else 0
#     avg_neg = sum(all_scores['negative']) / len(all_scores['negative']) if all_scores['negative'] else 0
#     avg_neu = sum(all_scores['neutral']) / len(all_scores['neutral']) if all_scores['neutral'] else 0

#     scores = {'positive': avg_pos, 'negative': avg_neg, 'neutral': avg_neu}
#     final_label = max(scores, key=scores.get)

#     return pd.Series({
#         'label': final_label,
#         'pos_score': avg_pos,
#         'neg_score': avg_neg,
#         'neutral_score': avg_neu
#     })

# # 5. Apply to your dataframe
# results = df['speech'].apply(get_sentiment)
# df = pd.concat([df, results], axis=1)

# print(df[['speech', 'label', 'pos_score', 'neg_score', 'neutral_score']].head())

In [11]:
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    return text #removes all numbers

In [12]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

# --- Step 1: Load BERT/RoBERTa model ---
model_name = "dost-asti/RoBERTa-tl-sentiment-analysis"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

sentiment_pipeline = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    top_k=None
)

label_map = {
    'label_0': 'negative',
    'label_1': 'positive',
    'label_2': 'neutral'
}

def chunk_text(text, tokenizer, max_length=510, stride=50):
    tokens = tokenizer.encode(text, add_special_tokens=False)
    chunks = []
    for i in range(0, len(tokens), max_length - stride):
        chunk_tokens = tokens[i:i + max_length]
        chunk_text = tokenizer.decode(chunk_tokens)
        chunks.append(chunk_text)
        if i + max_length >= len(tokens):
            break
    return chunks

def get_sentiment(text, tokenizer=tokenizer, pipe=sentiment_pipeline):
    if not isinstance(text, str) or not text.strip():
        return pd.Series({'label': 'neutral', 'pos_score': 0, 'neg_score': 0, 'neutral_score': 0})

    chunks = chunk_text(text, tokenizer)
    all_scores = {'positive': [], 'negative': [], 'neutral': []}

    for chunk in chunks:
        result = pipe(chunk, truncation=True, max_length=510)[0]
        for item in result:
            raw_label = item['label'].lower()
            mapped_label = label_map.get(raw_label, raw_label)
            if mapped_label in all_scores:
                all_scores[mapped_label].append(item['score'])

    avg_pos = sum(all_scores['positive']) / len(all_scores['positive']) if all_scores['positive'] else 0
    avg_neg = sum(all_scores['negative']) / len(all_scores['negative']) if all_scores['negative'] else 0
    avg_neu = sum(all_scores['neutral']) / len(all_scores['neutral']) if all_scores['neutral'] else 0

    scores = {'positive': avg_pos, 'negative': avg_neg, 'neutral': avg_neu}
    final_label = max(scores, key=scores.get)

    return pd.Series({
        'label': final_label,
        'pos_score': avg_pos,
        'neg_score': avg_neg,
        'neutral_score': avg_neu
    })

# --- Step 2: Run BERT sentiment on speeches ---
df['speech'] = df['speech'].fillna('')
bert_results = df['speech'].apply(get_sentiment)
df = pd.concat([df, bert_results], axis=1)

STPWORDS = stopwords.stopwords(["en", "tl"])  # base English + Tagalog stopwords
STPWORDS.update(['yung', 'iyan', 'yan', 'diyan', 'applause', 'cheers','laughter', 'palakpakan', 'rin', 'din', 'po',
                'pong', 'pang', 'pa', 'nang', 'ng', 'pag',
                'kapag', 'nga', 'upang','naman', 'natin', 'kayo',
                'nating', 'natin', 'tayong', 'lang', 'nag', 'kay', 'yang', 'iyan', 'yong', 'kasi'])

# --- Step 3: Run TF-IDF on the SAME speeches, using the correct stopword list ---
tfidf = TfidfVectorizer(
    stop_words=list(STPWORDS),   # <-- pass the actual word set, not language codes
    ngram_range=(1, 1),
    min_df=1,                     # min_df=0 is invalid for TfidfVectorizer, use 1 (or a small int/float)
    preprocessor=preprocess_text  # reuse the same text-cleaning function
)

tfidf_matrix = tfidf.fit_transform(df['speech'])

tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf.get_feature_names_out(),
    index=df.index
)

# --- Step 4: Extract top TF-IDF keywords per speech ---
def get_top_tfidf_words(row, top_n=10):
    return list(row.sort_values(ascending=False).head(top_n).index)

df['top_tfidf_words'] = tfidf_df.apply(get_top_tfidf_words, axis=1)

# --- Step 5: View combined results ---
print(df[['speech', 'top_tfidf_words', 'label', 'pos_score', 'neg_score', 'neutral_score']].head())

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer RobertaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
/opt/homebrew/lib/python3.10/site-packages/sklearn/feature_extraction/text.py:404: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['ain', 'daren', 'hadn', 'herse', 'himse', 'itse', 'mayn', 'mightn', 'mon', 'mustn', 'myse', 'needn', 'oughtn', 'shan'] not in stop_words.
  warnings.warn(


                                              speech  \
0   Mr. Speaker, gentlemen of the National Assemb...   
1   Mr. Speaker, Gentlemen of the National Assemb...   
2   Mr. Speaker, Gentlemen of the National Assemb...   
3   Gentlemen of the National Assembly: The state...   
4   Gentlemen of the National Assembly: I take pl...   

                                     top_tfidf_words     label  pos_score  \
0  [army, defense, military, defensive, training,...  negative   0.457971   
1  [government, railroad, national, commonwealth,...  negative   0.457538   
2  [independence, philippines, committee, governm...  negative   0.494213   
3  [tax, taxes, sales, income, government, paid, ...  negative   0.105746   
4  [assembly, national, recommendations, universi...  positive   0.822032   

   neg_score  neutral_score  
0   0.541995       0.000034  
1   0.538548       0.003914  
2   0.505611       0.000176  
3   0.806921       0.087333  
4   0.177949       0.000019  


In [13]:
df

,president,date,title,link,venue,session,speech,total_words,textblob,nltk,sentiment_label,label,pos_score,neg_score,neutral_score,top_tfidf_words
0,Manuel L. Quezon,"November 25, 1935",Message to the First Assembly on National Defense,https://www.officialgazette.gov.ph/1935/11/25/...,"Legislative Building, Manila","First National Assembly, First Session","Mr. Speaker, gentlemen of the National Assemb...",4341,0.106794,0.9999,positive,negative,0.457971,0.541995,0.000034,"[army, defense, military, defensive, training,..."
1,Manuel L. Quezon,"June 16, 1936",On the Country’s Conditions and Problems,https://www.officialgazette.gov.ph/1936/06/16/...,"Legislative Building, Manila","First National Assembly, First Session","Mr. Speaker, Gentlemen of the National Assemb...",7250,0.110458,1.0000,positive,negative,0.457538,0.538548,0.003914,"[government, railroad, national, commonwealth,..."
2,Manuel L. Quezon,"October 18, 1937","Improvement of Philippine Conditions, Philippi...",https://www.officialgazette.gov.ph/1937/10/18/...,"Legislative Building, Manila","First National Assembly, Second Session","Mr. Speaker, Gentlemen of the National Assemb...",5774,0.137388,1.0000,positive,negative,0.494213,0.505611,0.000176,"[independence, philippines, committee, governm..."
3,Manuel L. Quezon,"January 24, 1938",Revision of the System of Taxation,https://www.officialgazette.gov.ph/1938/01/24/...,"Legislative Building, Manila","First National Assembly, Third Session",Gentlemen of the National Assembly: The state...,3212,0.060580,0.9995,positive,negative,0.105746,0.806921,0.087333,"[tax, taxes, sales, income, government, paid, ..."
4,Manuel L. Quezon,"January 24, 1939",The State of the Nation and Important Economic...,https://www.officialgazette.gov.ph/1939/01/24/...,"Legislative Building, Manila","Second National Assembly, First Session",Gentlemen of the National Assembly: I take pl...,4826,0.121574,1.0000,positive,positive,0.822032,0.177949,0.000019,"[assembly, national, recommendations, universi..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
82,Rodrigo Roa Duterte,"July 26, 2021",Sixth State of the Nation Address,https://www.officialgazette.gov.ph/2021/07/26/...,"Batasang Pambansa, Quezon City","Eighteenth Congress, Third Session",Kindly sit down. By far this is the most bea...,14769,0.133088,1.0000,positive,negative,0.411494,0.588375,0.000131,"[covid, itong, pandemic, talaga, ninyo, ganoon..."
83,Ferdinand R. Marcos Jr.,"July 25, 2022",First State of the Nation Address,https://www.officialgazette.gov.ph/2021/07/26/...,"Batasang Pambansa, Quezon City","Nineteenth Congress, First Session",Vice President Sara Zimmerman Duterte; Form...,7990,0.119099,1.0000,positive,positive,0.997522,0.002124,0.000353,"[pandemic, seeks, covid, railway, department, ..."
84,Ferdinand R. Marcos Jr.,"July 24, 2023",Second State of the Nation Address,https://www.officialgazette.gov.ph/sf0clA,"Batasang Pambansa, Quezon City","Nineteenth Congress, Second Session",Thank you. Thank you very much. Allow me to g...,8227,0.139571,1.0000,positive,positive,0.999729,0.000248,0.000023,"[percent, mahigit, libong, government, buong, ..."
85,Ferdinand R. Marcos Jr.,"July 22, 2024",Third State of the Nation Address,https://www.officialgazette.gov.ph/YdHHeA,"Batasang Pambansa, Quezon City","Nineteenth Congress, Third Session",Thank you. Allow me to greet the former Presi...,9177,0.139383,1.0000,positive,positive,0.934042,0.037473,0.028485,"[libong, mahigit, bansa, taon, barmm, buong, t..."


In [14]:
df.label.value_counts()

positive    71
negative    15
Name: label, dtype: int64

## Looking into the results deeper

In [15]:
df[df.president=="Rodrigo Roa Duterte"]

,president,date,title,link,venue,session,speech,total_words,textblob,nltk,sentiment_label,label,pos_score,neg_score,neutral_score,top_tfidf_words
77,Rodrigo Roa Duterte,"July 25, 2016",State of the Nation Address,https://www.officialgazette.gov.ph/2016/07/25/...,"Batasang Pambansa, Quezon City","Seventeenth Congress, First Session",Thank you. Please allow me a little bit of ...,9306,0.082162,0.9998,positive,negative,0.476424,0.523088,0.000487,"[wala, ninyo, talaga, gina, wag, itong, alam, ..."
78,Rodrigo Roa Duterte,"July 24, 2017",Second State of the Nation Address,https://www.officialgazette.gov.ph/2017/07/24/...,"Batasang Pambansa, Quezon City","Seventeenth Congress, Second Session",Kindly sit down. Thank you for your courtes...,12712,0.055290,-0.9980,negative,negative,0.164665,0.835233,0.000102,"[ninyo, wala, ganun, kayong, itong, government..."
79,Rodrigo Roa Duterte,"July 23, 2018",Third State of the Nation Address,https://www.officialgazette.gov.ph/2018/07/23/...,"Batasang Pambansa, Quezon City","Seventeenth Congress, Third Session",Kindly sit down. Thank you for your courtesy....,4832,0.099200,0.9999,positive,positive,0.659441,0.340539,0.000019,"[government, people, businesses, filipino, ase..."
80,Rodrigo Roa Duterte,"July 22, 2019",Fourth State of the Nation Address,https://www.officialgazette.gov.ph/2019/07/22/...,"Batasang Pambansa, Quezon City","Eighteenth Congress, First Session",Thank you. Kindly sit down. Kumusta po kayo...,9537,0.070946,0.9990,positive,negative,0.288487,0.711423,0.000090,"[tsk, ninyo, wala, sir, talaga, pera, alam, mo..."
81,Rodrigo Roa Duterte,"July 27, 2020",Fifth State of the Nation Address,https://www.officialgazette.gov.ph/2020/07/27/...,"Batasang Pambansa, Quezon City","Eighteenth Congress, Second Session",Kindly… Senate President Vicente Sotto III an...,8763,0.106255,0.9999,positive,positive,0.510808,0.489145,0.000046,"[pandemic, covid, online, government, alam, ol..."
82,Rodrigo Roa Duterte,"July 26, 2021",Sixth State of the Nation Address,https://www.officialgazette.gov.ph/2021/07/26/...,"Batasang Pambansa, Quezon City","Eighteenth Congress, Third Session",Kindly sit down. By far this is the most bea...,14769,0.133088,1.0000,positive,negative,0.411494,0.588375,0.000131,"[covid, itong, pandemic, talaga, ninyo, ganoon..."


## Getting a weighted score

This is to be able to get some kind of polarity score for a speech.

In [16]:
df['overall_sentiment_score'] = df['pos_score'] - df['neg_score']
df

,president,date,title,link,venue,session,speech,total_words,textblob,nltk,sentiment_label,label,pos_score,neg_score,neutral_score,top_tfidf_words,overall_sentiment_score
0,Manuel L. Quezon,"November 25, 1935",Message to the First Assembly on National Defense,https://www.officialgazette.gov.ph/1935/11/25/...,"Legislative Building, Manila","First National Assembly, First Session","Mr. Speaker, gentlemen of the National Assemb...",4341,0.106794,0.9999,positive,negative,0.457971,0.541995,0.000034,"[army, defense, military, defensive, training,...",-0.084024
1,Manuel L. Quezon,"June 16, 1936",On the Country’s Conditions and Problems,https://www.officialgazette.gov.ph/1936/06/16/...,"Legislative Building, Manila","First National Assembly, First Session","Mr. Speaker, Gentlemen of the National Assemb...",7250,0.110458,1.0000,positive,negative,0.457538,0.538548,0.003914,"[government, railroad, national, commonwealth,...",-0.081010
2,Manuel L. Quezon,"October 18, 1937","Improvement of Philippine Conditions, Philippi...",https://www.officialgazette.gov.ph/1937/10/18/...,"Legislative Building, Manila","First National Assembly, Second Session","Mr. Speaker, Gentlemen of the National Assemb...",5774,0.137388,1.0000,positive,negative,0.494213,0.505611,0.000176,"[independence, philippines, committee, governm...",-0.011398
3,Manuel L. Quezon,"January 24, 1938",Revision of the System of Taxation,https://www.officialgazette.gov.ph/1938/01/24/...,"Legislative Building, Manila","First National Assembly, Third Session",Gentlemen of the National Assembly: The state...,3212,0.060580,0.9995,positive,negative,0.105746,0.806921,0.087333,"[tax, taxes, sales, income, government, paid, ...",-0.701175
4,Manuel L. Quezon,"January 24, 1939",The State of the Nation and Important Economic...,https://www.officialgazette.gov.ph/1939/01/24/...,"Legislative Building, Manila","Second National Assembly, First Session",Gentlemen of the National Assembly: I take pl...,4826,0.121574,1.0000,positive,positive,0.822032,0.177949,0.000019,"[assembly, national, recommendations, universi...",0.644082
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
82,Rodrigo Roa Duterte,"July 26, 2021",Sixth State of the Nation Address,https://www.officialgazette.gov.ph/2021/07/26/...,"Batasang Pambansa, Quezon City","Eighteenth Congress, Third Session",Kindly sit down. By far this is the most bea...,14769,0.133088,1.0000,positive,negative,0.411494,0.588375,0.000131,"[covid, itong, pandemic, talaga, ninyo, ganoon...",-0.176881
83,Ferdinand R. Marcos Jr.,"July 25, 2022",First State of the Nation Address,https://www.officialgazette.gov.ph/2021/07/26/...,"Batasang Pambansa, Quezon City","Nineteenth Congress, First Session",Vice President Sara Zimmerman Duterte; Form...,7990,0.119099,1.0000,positive,positive,0.997522,0.002124,0.000353,"[pandemic, seeks, covid, railway, department, ...",0.995398
84,Ferdinand R. Marcos Jr.,"July 24, 2023",Second State of the Nation Address,https://www.officialgazette.gov.ph/sf0clA,"Batasang Pambansa, Quezon City","Nineteenth Congress, Second Session",Thank you. Thank you very much. Allow me to g...,8227,0.139571,1.0000,positive,positive,0.999729,0.000248,0.000023,"[percent, mahigit, libong, government, buong, ...",0.999481
85,Ferdinand R. Marcos Jr.,"July 22, 2024",Third State of the Nation Address,https://www.officialgazette.gov.ph/YdHHeA,"Batasang Pambansa, Quezon City","Nineteenth Congress, Third Session",Thank you. Allow me to greet the former Presi...,9177,0.139383,1.0000,positive,positive,0.934042,0.037473,0.028485,"[libong, mahigit, bansa, taon, barmm, buong, t...",0.896570


In [17]:
df['total_score'] = df['pos_score'] + df['neg_score'] + df['neutral_score']
df

##not useful but just to check that all is equal to 1.

,president,date,title,link,venue,session,speech,total_words,textblob,nltk,sentiment_label,label,pos_score,neg_score,neutral_score,top_tfidf_words,overall_sentiment_score,total_score
0,Manuel L. Quezon,"November 25, 1935",Message to the First Assembly on National Defense,https://www.officialgazette.gov.ph/1935/11/25/...,"Legislative Building, Manila","First National Assembly, First Session","Mr. Speaker, gentlemen of the National Assemb...",4341,0.106794,0.9999,positive,negative,0.457971,0.541995,0.000034,"[army, defense, military, defensive, training,...",-0.084024,1.0
1,Manuel L. Quezon,"June 16, 1936",On the Country’s Conditions and Problems,https://www.officialgazette.gov.ph/1936/06/16/...,"Legislative Building, Manila","First National Assembly, First Session","Mr. Speaker, Gentlemen of the National Assemb...",7250,0.110458,1.0000,positive,negative,0.457538,0.538548,0.003914,"[government, railroad, national, commonwealth,...",-0.081010,1.0
2,Manuel L. Quezon,"October 18, 1937","Improvement of Philippine Conditions, Philippi...",https://www.officialgazette.gov.ph/1937/10/18/...,"Legislative Building, Manila","First National Assembly, Second Session","Mr. Speaker, Gentlemen of the National Assemb...",5774,0.137388,1.0000,positive,negative,0.494213,0.505611,0.000176,"[independence, philippines, committee, governm...",-0.011398,1.0
3,Manuel L. Quezon,"January 24, 1938",Revision of the System of Taxation,https://www.officialgazette.gov.ph/1938/01/24/...,"Legislative Building, Manila","First National Assembly, Third Session",Gentlemen of the National Assembly: The state...,3212,0.060580,0.9995,positive,negative,0.105746,0.806921,0.087333,"[tax, taxes, sales, income, government, paid, ...",-0.701175,1.0
4,Manuel L. Quezon,"January 24, 1939",The State of the Nation and Important Economic...,https://www.officialgazette.gov.ph/1939/01/24/...,"Legislative Building, Manila","Second National Assembly, First Session",Gentlemen of the National Assembly: I take pl...,4826,0.121574,1.0000,positive,positive,0.822032,0.177949,0.000019,"[assembly, national, recommendations, universi...",0.644082,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
82,Rodrigo Roa Duterte,"July 26, 2021",Sixth State of the Nation Address,https://www.officialgazette.gov.ph/2021/07/26/...,"Batasang Pambansa, Quezon City","Eighteenth Congress, Third Session",Kindly sit down. By far this is the most bea...,14769,0.133088,1.0000,positive,negative,0.411494,0.588375,0.000131,"[covid, itong, pandemic, talaga, ninyo, ganoon...",-0.176881,1.0
83,Ferdinand R. Marcos Jr.,"July 25, 2022",First State of the Nation Address,https://www.officialgazette.gov.ph/2021/07/26/...,"Batasang Pambansa, Quezon City","Nineteenth Congress, First Session",Vice President Sara Zimmerman Duterte; Form...,7990,0.119099,1.0000,positive,positive,0.997522,0.002124,0.000353,"[pandemic, seeks, covid, railway, department, ...",0.995398,1.0
84,Ferdinand R. Marcos Jr.,"July 24, 2023",Second State of the Nation Address,https://www.officialgazette.gov.ph/sf0clA,"Batasang Pambansa, Quezon City","Nineteenth Congress, Second Session",Thank you. Thank you very much. Allow me to g...,8227,0.139571,1.0000,positive,positive,0.999729,0.000248,0.000023,"[percent, mahigit, libong, government, buong, ...",0.999481,1.0
85,Ferdinand R. Marcos Jr.,"July 22, 2024",Third State of the Nation Address,https://www.officialgazette.gov.ph/YdHHeA,"Batasang Pambansa, Quezon City","Nineteenth Congress, Third Session",Thank you. Allow me to greet the former Presi...,9177,0.139383,1.0000,positive,positive,0.934042,0.037473,0.028485,"[libong, mahigit, bansa, taon, barmm, buong, t...",0.896570,1.0


## Check similarities/differences between lexicon-based sentiment and RoBERTa

In [18]:
df = df.rename(columns={'label': 'roberta_label'})
df['matched_results'] = df['sentiment_label'] == df['roberta_label']
df

,president,date,title,link,venue,session,speech,total_words,textblob,nltk,sentiment_label,roberta_label,pos_score,neg_score,neutral_score,top_tfidf_words,overall_sentiment_score,total_score,matched_results
0,Manuel L. Quezon,"November 25, 1935",Message to the First Assembly on National Defense,https://www.officialgazette.gov.ph/1935/11/25/...,"Legislative Building, Manila","First National Assembly, First Session","Mr. Speaker, gentlemen of the National Assemb...",4341,0.106794,0.9999,positive,negative,0.457971,0.541995,0.000034,"[army, defense, military, defensive, training,...",-0.084024,1.0,False
1,Manuel L. Quezon,"June 16, 1936",On the Country’s Conditions and Problems,https://www.officialgazette.gov.ph/1936/06/16/...,"Legislative Building, Manila","First National Assembly, First Session","Mr. Speaker, Gentlemen of the National Assemb...",7250,0.110458,1.0000,positive,negative,0.457538,0.538548,0.003914,"[government, railroad, national, commonwealth,...",-0.081010,1.0,False
2,Manuel L. Quezon,"October 18, 1937","Improvement of Philippine Conditions, Philippi...",https://www.officialgazette.gov.ph/1937/10/18/...,"Legislative Building, Manila","First National Assembly, Second Session","Mr. Speaker, Gentlemen of the National Assemb...",5774,0.137388,1.0000,positive,negative,0.494213,0.505611,0.000176,"[independence, philippines, committee, governm...",-0.011398,1.0,False
3,Manuel L. Quezon,"January 24, 1938",Revision of the System of Taxation,https://www.officialgazette.gov.ph/1938/01/24/...,"Legislative Building, Manila","First National Assembly, Third Session",Gentlemen of the National Assembly: The state...,3212,0.060580,0.9995,positive,negative,0.105746,0.806921,0.087333,"[tax, taxes, sales, income, government, paid, ...",-0.701175,1.0,False
4,Manuel L. Quezon,"January 24, 1939",The State of the Nation and Important Economic...,https://www.officialgazette.gov.ph/1939/01/24/...,"Legislative Building, Manila","Second National Assembly, First Session",Gentlemen of the National Assembly: I take pl...,4826,0.121574,1.0000,positive,positive,0.822032,0.177949,0.000019,"[assembly, national, recommendations, universi...",0.644082,1.0,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
82,Rodrigo Roa Duterte,"July 26, 2021",Sixth State of the Nation Address,https://www.officialgazette.gov.ph/2021/07/26/...,"Batasang Pambansa, Quezon City","Eighteenth Congress, Third Session",Kindly sit down. By far this is the most bea...,14769,0.133088,1.0000,positive,negative,0.411494,0.588375,0.000131,"[covid, itong, pandemic, talaga, ninyo, ganoon...",-0.176881,1.0,False
83,Ferdinand R. Marcos Jr.,"July 25, 2022",First State of the Nation Address,https://www.officialgazette.gov.ph/2021/07/26/...,"Batasang Pambansa, Quezon City","Nineteenth Congress, First Session",Vice President Sara Zimmerman Duterte; Form...,7990,0.119099,1.0000,positive,positive,0.997522,0.002124,0.000353,"[pandemic, seeks, covid, railway, department, ...",0.995398,1.0,True
84,Ferdinand R. Marcos Jr.,"July 24, 2023",Second State of the Nation Address,https://www.officialgazette.gov.ph/sf0clA,"Batasang Pambansa, Quezon City","Nineteenth Congress, Second Session",Thank you. Thank you very much. Allow me to g...,8227,0.139571,1.0000,positive,positive,0.999729,0.000248,0.000023,"[percent, mahigit, libong, government, buong, ...",0.999481,1.0,True
85,Ferdinand R. Marcos Jr.,"July 22, 2024",Third State of the Nation Address,https://www.officialgazette.gov.ph/YdHHeA,"Batasang Pambansa, Quezon City","Nineteenth Congress, Third Session",Thank you. Allow me to greet the former Presi...,9177,0.139383,1.0000,positive,positive,0.934042,0.037473,0.028485,"[libong, mahigit, bansa, taon, barmm, buong, t...",0.896570,1.0,True


In [19]:
df.matched_results.value_counts()

True     67
False    19
Name: matched_results, dtype: int64

In [20]:
df[df.matched_results==False]

,president,date,title,link,venue,session,speech,total_words,textblob,nltk,sentiment_label,roberta_label,pos_score,neg_score,neutral_score,top_tfidf_words,overall_sentiment_score,total_score,matched_results
0,Manuel L. Quezon,"November 25, 1935",Message to the First Assembly on National Defense,https://www.officialgazette.gov.ph/1935/11/25/...,"Legislative Building, Manila","First National Assembly, First Session","Mr. Speaker, gentlemen of the National Assemb...",4341,0.106794,0.9999,positive,negative,0.457971,0.541995,0.000034,"[army, defense, military, defensive, training,...",-0.084024,1.0,False
1,Manuel L. Quezon,"June 16, 1936",On the Country’s Conditions and Problems,https://www.officialgazette.gov.ph/1936/06/16/...,"Legislative Building, Manila","First National Assembly, First Session","Mr. Speaker, Gentlemen of the National Assemb...",7250,0.110458,1.0000,positive,negative,0.457538,0.538548,0.003914,"[government, railroad, national, commonwealth,...",-0.081010,1.0,False
2,Manuel L. Quezon,"October 18, 1937","Improvement of Philippine Conditions, Philippi...",https://www.officialgazette.gov.ph/1937/10/18/...,"Legislative Building, Manila","First National Assembly, Second Session","Mr. Speaker, Gentlemen of the National Assemb...",5774,0.137388,1.0000,positive,negative,0.494213,0.505611,0.000176,"[independence, philippines, committee, governm...",-0.011398,1.0,False
3,Manuel L. Quezon,"January 24, 1938",Revision of the System of Taxation,https://www.officialgazette.gov.ph/1938/01/24/...,"Legislative Building, Manila","First National Assembly, Third Session",Gentlemen of the National Assembly: The state...,3212,0.060580,0.9995,positive,negative,0.105746,0.806921,0.087333,"[tax, taxes, sales, income, government, paid, ...",-0.701175,1.0,False
5,Manuel L. Quezon,"January 22, 1940",The State of the Nation,https://www.officialgazette.gov.ph/1940/01/22/...,"Legislative Building, Manila","Second National Assembly, Second Session",Gentlemen of the National Assembly: You are c...,5838,0.100285,0.9999,positive,negative,0.470679,0.529245,0.000075,"[government, national, assembly, owners, tobac...",-0.058566,1.0,False
36,Ferdinand E. Marcos,"September 21, 1974",The Barangay and the Imperative of National Unity,https://www.officialgazette.gov.ph/1974/09/21/...,"Maharlika Hall, Malacañan Palace",NaN,“The Barangay and the Imperative of National ...,9136,0.093389,0.9998,positive,negative,0.469584,0.530274,0.000142,"[barangays, samahang, barangay, nayon, addicti...",-0.060690,1.0,False
37,Ferdinand E. Marcos,"September 19, 1975",The President’s Report to the Nation,https://www.officialgazette.gov.ph/1975/09/19/...,Quirino Grandstand,NaN,“The President’s Report to the Nation” Itinak...,6099,0.069883,0.9998,positive,negative,0.441216,0.558754,0.000031,"[resignation, government, accept, society, res...",-0.117538,1.0,False
44,Ferdinand E. Marcos,"July 26, 1982",State-of-the-Nation Address,https://www.officialgazette.gov.ph/1982/07/26/...,"Batasang Pambansa, Quezon City",Interim Batasang Pambansa,The President; Mr. Speaker; distinguished col...,5065,0.072171,0.9995,positive,negative,0.219172,0.762507,0.018321,"[batasang, pambansa, constitution, government,...",-0.543334,1.0,False
52,Corazon C. Aquino,"July 22, 1991",The State of the Nation,https://www.officialgazette.gov.ph/1991/07/22/...,"Batasang Pambansa, Quezon City","Eighth Congress, Fifth Session",Senate President Jovito Salonga; Speaker Ramo...,4001,0.127536,0.9999,positive,negative,0.394264,0.605711,0.000025,"[people, elections, government, candle, democr...",-0.211447,1.0,False
59,Joseph Ejercito Estrada,"July 27, 1998",The State of the Nation,https://www.officialgazette.gov.ph/1998/07/27/...,"Batasang Pambansa, Quezon City","Eleventh Congress, First Session",Ang Pangalawang Pangulo ng Republika ng Pilip...,3981,0.065895,0.9904,positive,negative,0.208766,0.791217,0.000018,"[percent, sawa, kagalang, galang, government, ...",-0.582451,1.0,False


## Aquino speech English translation

Clean it first before merging

In [21]:
aquino_english = pd.read_csv('./csv/aquino_english_scraped.csv')
aquino_english

,president,date,title,link,venue,speech_text
0,Benigno S. Aquino III,26-Jul-10,State of the Nation Address,https://www.officialgazette.gov.ph/2010/07/26/...,"Batasang Pambansa, Quezon City",State of the Nation Address of His Excellency\...
1,Benigno S. Aquino III,25-Jul-11,Second State of the Nation Address,https://www.officialgazette.gov.ph/2011/07/25/...,"Batasang Pambansa, Quezon City",State of the Nation Address\nof \nHis Excellen...
2,Benigno S. Aquino III,23-Jul-12,Third State of the Nation Address,https://www.officialgazette.gov.ph/2012/07/23/...,"Batasang Pambansa, Quezon City",State of the Nation Address\nof\nHis Excellenc...
3,Benigno S. Aquino III,22-Jul-13,Fourth State of the Nation Address,https://www.officialgazette.gov.ph/2013/07/22/...,"Batasang Pambansa, Quezon City",State of the Nation Address\nof\nHis Excellenc...
4,Benigno S. Aquino III,28-Jul-14,Fifth State of the Nation Address,https://www.officialgazette.gov.ph/2014/07/28/...,"Batasang Pambansa, Quezon City",State of the Nation Address\nof\nHis Excellenc...
5,Benigno S. Aquino III,27-Jul-15,Sixth State of the Nation Address,https://www.officialgazette.gov.ph/2015/07/27/...,"Batasang Pambansa, Quezon City",The 2015 State of the Nation Address\n[Basahin...


In [22]:
aquino_english = aquino_english.rename(columns={'speech_text': 'speech'})
aquino_english

,president,date,title,link,venue,speech
0,Benigno S. Aquino III,26-Jul-10,State of the Nation Address,https://www.officialgazette.gov.ph/2010/07/26/...,"Batasang Pambansa, Quezon City",State of the Nation Address of His Excellency\...
1,Benigno S. Aquino III,25-Jul-11,Second State of the Nation Address,https://www.officialgazette.gov.ph/2011/07/25/...,"Batasang Pambansa, Quezon City",State of the Nation Address\nof \nHis Excellen...
2,Benigno S. Aquino III,23-Jul-12,Third State of the Nation Address,https://www.officialgazette.gov.ph/2012/07/23/...,"Batasang Pambansa, Quezon City",State of the Nation Address\nof\nHis Excellenc...
3,Benigno S. Aquino III,22-Jul-13,Fourth State of the Nation Address,https://www.officialgazette.gov.ph/2013/07/22/...,"Batasang Pambansa, Quezon City",State of the Nation Address\nof\nHis Excellenc...
4,Benigno S. Aquino III,28-Jul-14,Fifth State of the Nation Address,https://www.officialgazette.gov.ph/2014/07/28/...,"Batasang Pambansa, Quezon City",State of the Nation Address\nof\nHis Excellenc...
5,Benigno S. Aquino III,27-Jul-15,Sixth State of the Nation Address,https://www.officialgazette.gov.ph/2015/07/27/...,"Batasang Pambansa, Quezon City",The 2015 State of the Nation Address\n[Basahin...


In [23]:
aquino_english.speech = aquino_english.speech.str.replace('\n', ' ', regex=True)
aquino_english

,president,date,title,link,venue,speech
0,Benigno S. Aquino III,26-Jul-10,State of the Nation Address,https://www.officialgazette.gov.ph/2010/07/26/...,"Batasang Pambansa, Quezon City",State of the Nation Address of His Excellency ...
1,Benigno S. Aquino III,25-Jul-11,Second State of the Nation Address,https://www.officialgazette.gov.ph/2011/07/25/...,"Batasang Pambansa, Quezon City",State of the Nation Address of His Excellency...
2,Benigno S. Aquino III,23-Jul-12,Third State of the Nation Address,https://www.officialgazette.gov.ph/2012/07/23/...,"Batasang Pambansa, Quezon City",State of the Nation Address of His Excellency ...
3,Benigno S. Aquino III,22-Jul-13,Fourth State of the Nation Address,https://www.officialgazette.gov.ph/2013/07/22/...,"Batasang Pambansa, Quezon City",State of the Nation Address of His Excellency ...
4,Benigno S. Aquino III,28-Jul-14,Fifth State of the Nation Address,https://www.officialgazette.gov.ph/2014/07/28/...,"Batasang Pambansa, Quezon City",State of the Nation Address of His Excellency ...
5,Benigno S. Aquino III,27-Jul-15,Sixth State of the Nation Address,https://www.officialgazette.gov.ph/2015/07/27/...,"Batasang Pambansa, Quezon City",The 2015 State of the Nation Address [Basahin ...


In [24]:
aquino_english.speech = aquino_english.speech.str.replace(r"^.+?(?=English)", "", regex=True)
aquino_english

,president,date,title,link,venue,speech
0,Benigno S. Aquino III,26-Jul-10,State of the Nation Address,https://www.officialgazette.gov.ph/2010/07/26/...,"Batasang Pambansa, Quezon City",State of the Nation Address of His Excellency ...
1,Benigno S. Aquino III,25-Jul-11,Second State of the Nation Address,https://www.officialgazette.gov.ph/2011/07/25/...,"Batasang Pambansa, Quezon City",English translation of the speech delivered at...
2,Benigno S. Aquino III,23-Jul-12,Third State of the Nation Address,https://www.officialgazette.gov.ph/2012/07/23/...,"Batasang Pambansa, Quezon City",English translation of the speech delivered at...
3,Benigno S. Aquino III,22-Jul-13,Fourth State of the Nation Address,https://www.officialgazette.gov.ph/2013/07/22/...,"Batasang Pambansa, Quezon City",English translation of the SONA delivered at t...
4,Benigno S. Aquino III,28-Jul-14,Fifth State of the Nation Address,https://www.officialgazette.gov.ph/2014/07/28/...,"Batasang Pambansa, Quezon City",English translation of the speech delivered at...
5,Benigno S. Aquino III,27-Jul-15,Sixth State of the Nation Address,https://www.officialgazette.gov.ph/2015/07/27/...,"Batasang Pambansa, Quezon City",English translation of the speech delivered at...


In [25]:
aquino_english.speech = aquino_english.speech.str.replace(r"^.+?(?=July)", "", regex=True)
aquino_english

,president,date,title,link,venue,speech
0,Benigno S. Aquino III,26-Jul-10,State of the Nation Address,https://www.officialgazette.gov.ph/2010/07/26/...,"Batasang Pambansa, Quezon City","July 26, 2010, Batasan Pambansa Complex, Quezo..."
1,Benigno S. Aquino III,25-Jul-11,Second State of the Nation Address,https://www.officialgazette.gov.ph/2011/07/25/...,"Batasang Pambansa, Quezon City","July 25, 2011] Senate President Juan Ponce Enr..."
2,Benigno S. Aquino III,23-Jul-12,Third State of the Nation Address,https://www.officialgazette.gov.ph/2012/07/23/...,"Batasang Pambansa, Quezon City","July 23, 2012] Senate President Juan Ponce Enr..."
3,Benigno S. Aquino III,22-Jul-13,Fourth State of the Nation Address,https://www.officialgazette.gov.ph/2013/07/22/...,"Batasang Pambansa, Quezon City","July 22, 2013] Vice President Jejomar Binay; S..."
4,Benigno S. Aquino III,28-Jul-14,Fifth State of the Nation Address,https://www.officialgazette.gov.ph/2014/07/28/...,"Batasang Pambansa, Quezon City","July 28, 2014] Vice President Jejomar Binay; P..."
5,Benigno S. Aquino III,27-Jul-15,Sixth State of the Nation Address,https://www.officialgazette.gov.ph/2015/07/27/...,"Batasang Pambansa, Quezon City","July 27, 2015] Thank you, everyone. Please sit..."


In [26]:
aquino_english.speech = aquino_english.speech.str.replace(r"^.+?(?=])", "", regex=True)
aquino_english

,president,date,title,link,venue,speech
0,Benigno S. Aquino III,26-Jul-10,State of the Nation Address,https://www.officialgazette.gov.ph/2010/07/26/...,"Batasang Pambansa, Quezon City",] Speaker Feliciano Belmonte; Senate President...
1,Benigno S. Aquino III,25-Jul-11,Second State of the Nation Address,https://www.officialgazette.gov.ph/2011/07/25/...,"Batasang Pambansa, Quezon City",] Senate President Juan Ponce Enrile; Speaker ...
2,Benigno S. Aquino III,23-Jul-12,Third State of the Nation Address,https://www.officialgazette.gov.ph/2012/07/23/...,"Batasang Pambansa, Quezon City",] Senate President Juan Ponce Enrile; Speaker ...
3,Benigno S. Aquino III,22-Jul-13,Fourth State of the Nation Address,https://www.officialgazette.gov.ph/2013/07/22/...,"Batasang Pambansa, Quezon City",] Vice President Jejomar Binay; Senate Preside...
4,Benigno S. Aquino III,28-Jul-14,Fifth State of the Nation Address,https://www.officialgazette.gov.ph/2014/07/28/...,"Batasang Pambansa, Quezon City",] Vice President Jejomar Binay; President Fide...
5,Benigno S. Aquino III,27-Jul-15,Sixth State of the Nation Address,https://www.officialgazette.gov.ph/2015/07/27/...,"Batasang Pambansa, Quezon City","] Thank you, everyone. Please sit down. Before..."


In [27]:
aquino_english.speech = aquino_english.speech.str.replace(r'^]', "", regex=True)
aquino_english

,president,date,title,link,venue,speech
0,Benigno S. Aquino III,26-Jul-10,State of the Nation Address,https://www.officialgazette.gov.ph/2010/07/26/...,"Batasang Pambansa, Quezon City",Speaker Feliciano Belmonte; Senate President ...
1,Benigno S. Aquino III,25-Jul-11,Second State of the Nation Address,https://www.officialgazette.gov.ph/2011/07/25/...,"Batasang Pambansa, Quezon City",Senate President Juan Ponce Enrile; Speaker F...
2,Benigno S. Aquino III,23-Jul-12,Third State of the Nation Address,https://www.officialgazette.gov.ph/2012/07/23/...,"Batasang Pambansa, Quezon City",Senate President Juan Ponce Enrile; Speaker F...
3,Benigno S. Aquino III,22-Jul-13,Fourth State of the Nation Address,https://www.officialgazette.gov.ph/2013/07/22/...,"Batasang Pambansa, Quezon City",Vice President Jejomar Binay; Senate Presiden...
4,Benigno S. Aquino III,28-Jul-14,Fifth State of the Nation Address,https://www.officialgazette.gov.ph/2014/07/28/...,"Batasang Pambansa, Quezon City",Vice President Jejomar Binay; President Fidel...
5,Benigno S. Aquino III,27-Jul-15,Sixth State of the Nation Address,https://www.officialgazette.gov.ph/2015/07/27/...,"Batasang Pambansa, Quezon City","Thank you, everyone. Please sit down. Before ..."


In [28]:
aquino_english.president = aquino_english.president.fillna(method="ffill")
aquino_english

,president,date,title,link,venue,speech
0,Benigno S. Aquino III,26-Jul-10,State of the Nation Address,https://www.officialgazette.gov.ph/2010/07/26/...,"Batasang Pambansa, Quezon City",Speaker Feliciano Belmonte; Senate President ...
1,Benigno S. Aquino III,25-Jul-11,Second State of the Nation Address,https://www.officialgazette.gov.ph/2011/07/25/...,"Batasang Pambansa, Quezon City",Senate President Juan Ponce Enrile; Speaker F...
2,Benigno S. Aquino III,23-Jul-12,Third State of the Nation Address,https://www.officialgazette.gov.ph/2012/07/23/...,"Batasang Pambansa, Quezon City",Senate President Juan Ponce Enrile; Speaker F...
3,Benigno S. Aquino III,22-Jul-13,Fourth State of the Nation Address,https://www.officialgazette.gov.ph/2013/07/22/...,"Batasang Pambansa, Quezon City",Vice President Jejomar Binay; Senate Presiden...
4,Benigno S. Aquino III,28-Jul-14,Fifth State of the Nation Address,https://www.officialgazette.gov.ph/2014/07/28/...,"Batasang Pambansa, Quezon City",Vice President Jejomar Binay; President Fidel...
5,Benigno S. Aquino III,27-Jul-15,Sixth State of the Nation Address,https://www.officialgazette.gov.ph/2015/07/27/...,"Batasang Pambansa, Quezon City","Thank you, everyone. Please sit down. Before ..."


In [29]:
# aquino_english.to_csv('aquino_english_clean.csv', index=False)

## Merge

In [30]:
df

,president,date,title,link,venue,session,speech,total_words,textblob,nltk,sentiment_label,roberta_label,pos_score,neg_score,neutral_score,top_tfidf_words,overall_sentiment_score,total_score,matched_results
0,Manuel L. Quezon,"November 25, 1935",Message to the First Assembly on National Defense,https://www.officialgazette.gov.ph/1935/11/25/...,"Legislative Building, Manila","First National Assembly, First Session","Mr. Speaker, gentlemen of the National Assemb...",4341,0.106794,0.9999,positive,negative,0.457971,0.541995,0.000034,"[army, defense, military, defensive, training,...",-0.084024,1.0,False
1,Manuel L. Quezon,"June 16, 1936",On the Country’s Conditions and Problems,https://www.officialgazette.gov.ph/1936/06/16/...,"Legislative Building, Manila","First National Assembly, First Session","Mr. Speaker, Gentlemen of the National Assemb...",7250,0.110458,1.0000,positive,negative,0.457538,0.538548,0.003914,"[government, railroad, national, commonwealth,...",-0.081010,1.0,False
2,Manuel L. Quezon,"October 18, 1937","Improvement of Philippine Conditions, Philippi...",https://www.officialgazette.gov.ph/1937/10/18/...,"Legislative Building, Manila","First National Assembly, Second Session","Mr. Speaker, Gentlemen of the National Assemb...",5774,0.137388,1.0000,positive,negative,0.494213,0.505611,0.000176,"[independence, philippines, committee, governm...",-0.011398,1.0,False
3,Manuel L. Quezon,"January 24, 1938",Revision of the System of Taxation,https://www.officialgazette.gov.ph/1938/01/24/...,"Legislative Building, Manila","First National Assembly, Third Session",Gentlemen of the National Assembly: The state...,3212,0.060580,0.9995,positive,negative,0.105746,0.806921,0.087333,"[tax, taxes, sales, income, government, paid, ...",-0.701175,1.0,False
4,Manuel L. Quezon,"January 24, 1939",The State of the Nation and Important Economic...,https://www.officialgazette.gov.ph/1939/01/24/...,"Legislative Building, Manila","Second National Assembly, First Session",Gentlemen of the National Assembly: I take pl...,4826,0.121574,1.0000,positive,positive,0.822032,0.177949,0.000019,"[assembly, national, recommendations, universi...",0.644082,1.0,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
82,Rodrigo Roa Duterte,"July 26, 2021",Sixth State of the Nation Address,https://www.officialgazette.gov.ph/2021/07/26/...,"Batasang Pambansa, Quezon City","Eighteenth Congress, Third Session",Kindly sit down. By far this is the most bea...,14769,0.133088,1.0000,positive,negative,0.411494,0.588375,0.000131,"[covid, itong, pandemic, talaga, ninyo, ganoon...",-0.176881,1.0,False
83,Ferdinand R. Marcos Jr.,"July 25, 2022",First State of the Nation Address,https://www.officialgazette.gov.ph/2021/07/26/...,"Batasang Pambansa, Quezon City","Nineteenth Congress, First Session",Vice President Sara Zimmerman Duterte; Form...,7990,0.119099,1.0000,positive,positive,0.997522,0.002124,0.000353,"[pandemic, seeks, covid, railway, department, ...",0.995398,1.0,True
84,Ferdinand R. Marcos Jr.,"July 24, 2023",Second State of the Nation Address,https://www.officialgazette.gov.ph/sf0clA,"Batasang Pambansa, Quezon City","Nineteenth Congress, Second Session",Thank you. Thank you very much. Allow me to g...,8227,0.139571,1.0000,positive,positive,0.999729,0.000248,0.000023,"[percent, mahigit, libong, government, buong, ...",0.999481,1.0,True
85,Ferdinand R. Marcos Jr.,"July 22, 2024",Third State of the Nation Address,https://www.officialgazette.gov.ph/YdHHeA,"Batasang Pambansa, Quezon City","Nineteenth Congress, Third Session",Thank you. Allow me to greet the former Presi...,9177,0.139383,1.0000,positive,positive,0.934042,0.037473,0.028485,"[libong, mahigit, bansa, taon, barmm, buong, t...",0.896570,1.0,True


In [31]:
def extract_year(link):
    match = re.search(r'/(\d{4})/', str(link))
    return match.group(1) if match else None

# Extract year from link to create a unique key per speech
df['year'] = df['link'].apply(extract_year)
aquino_english['year'] = aquino_english['link'].apply(extract_year)

# Merge dfB's speech in as a temp column, matched on president + year
df = df.merge(
    aquino_english[['president', 'year', 'speech']],
    on=['president', 'year'],
    how='left',
    suffixes=('', '_fromB')
)

# Replace speech only where a match exists; everything else stays untouched
mask = df['speech_fromB'].notna()
df.loc[mask, 'speech'] = df.loc[mask, 'speech_fromB']

# Clean up helper columns
df = df.drop(columns=['speech_fromB', 'year'])
df

,president,date,title,link,venue,session,speech,total_words,textblob,nltk,sentiment_label,roberta_label,pos_score,neg_score,neutral_score,top_tfidf_words,overall_sentiment_score,total_score,matched_results
0,Manuel L. Quezon,"November 25, 1935",Message to the First Assembly on National Defense,https://www.officialgazette.gov.ph/1935/11/25/...,"Legislative Building, Manila","First National Assembly, First Session","Mr. Speaker, gentlemen of the National Assemb...",4341,0.106794,0.9999,positive,negative,0.457971,0.541995,0.000034,"[army, defense, military, defensive, training,...",-0.084024,1.0,False
1,Manuel L. Quezon,"June 16, 1936",On the Country’s Conditions and Problems,https://www.officialgazette.gov.ph/1936/06/16/...,"Legislative Building, Manila","First National Assembly, First Session","Mr. Speaker, Gentlemen of the National Assemb...",7250,0.110458,1.0000,positive,negative,0.457538,0.538548,0.003914,"[government, railroad, national, commonwealth,...",-0.081010,1.0,False
2,Manuel L. Quezon,"October 18, 1937","Improvement of Philippine Conditions, Philippi...",https://www.officialgazette.gov.ph/1937/10/18/...,"Legislative Building, Manila","First National Assembly, Second Session","Mr. Speaker, Gentlemen of the National Assemb...",5774,0.137388,1.0000,positive,negative,0.494213,0.505611,0.000176,"[independence, philippines, committee, governm...",-0.011398,1.0,False
3,Manuel L. Quezon,"January 24, 1938",Revision of the System of Taxation,https://www.officialgazette.gov.ph/1938/01/24/...,"Legislative Building, Manila","First National Assembly, Third Session",Gentlemen of the National Assembly: The state...,3212,0.060580,0.9995,positive,negative,0.105746,0.806921,0.087333,"[tax, taxes, sales, income, government, paid, ...",-0.701175,1.0,False
4,Manuel L. Quezon,"January 24, 1939",The State of the Nation and Important Economic...,https://www.officialgazette.gov.ph/1939/01/24/...,"Legislative Building, Manila","Second National Assembly, First Session",Gentlemen of the National Assembly: I take pl...,4826,0.121574,1.0000,positive,positive,0.822032,0.177949,0.000019,"[assembly, national, recommendations, universi...",0.644082,1.0,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81,Rodrigo Roa Duterte,"July 26, 2021",Sixth State of the Nation Address,https://www.officialgazette.gov.ph/2021/07/26/...,"Batasang Pambansa, Quezon City","Eighteenth Congress, Third Session",Kindly sit down. By far this is the most bea...,14769,0.133088,1.0000,positive,negative,0.411494,0.588375,0.000131,"[covid, itong, pandemic, talaga, ninyo, ganoon...",-0.176881,1.0,False
82,Ferdinand R. Marcos Jr.,"July 25, 2022",First State of the Nation Address,https://www.officialgazette.gov.ph/2021/07/26/...,"Batasang Pambansa, Quezon City","Nineteenth Congress, First Session",Vice President Sara Zimmerman Duterte; Form...,7990,0.119099,1.0000,positive,positive,0.997522,0.002124,0.000353,"[pandemic, seeks, covid, railway, department, ...",0.995398,1.0,True
83,Ferdinand R. Marcos Jr.,"July 24, 2023",Second State of the Nation Address,https://www.officialgazette.gov.ph/sf0clA,"Batasang Pambansa, Quezon City","Nineteenth Congress, Second Session",Thank you. Thank you very much. Allow me to g...,8227,0.139571,1.0000,positive,positive,0.999729,0.000248,0.000023,"[percent, mahigit, libong, government, buong, ...",0.999481,1.0,True
84,Ferdinand R. Marcos Jr.,"July 22, 2024",Third State of the Nation Address,https://www.officialgazette.gov.ph/YdHHeA,"Batasang Pambansa, Quezon City","Nineteenth Congress, Third Session",Thank you. Allow me to greet the former Presi...,9177,0.139383,1.0000,positive,positive,0.934042,0.037473,0.028485,"[libong, mahigit, bansa, taon, barmm, buong, t...",0.896570,1.0,True


In [32]:
df[df.president=="Benigno S. Aquino III"]

,president,date,title,link,venue,session,speech,total_words,textblob,nltk,sentiment_label,roberta_label,pos_score,neg_score,neutral_score,top_tfidf_words,overall_sentiment_score,total_score,matched_results
70,Benigno S. Aquino III,"July 26, 2010",State of the Nation Address,https://www.officialgazette.gov.ph/2010/07/26/...,"Batasang Pambansa, Quezon City","Fifteenth Congress, First Session",Speaker Feliciano Belmonte; Senate President ...,3828,-0.023173,-0.9998,negative,positive,0.494378,0.475927,0.029696,"[pesos, taumbayan, porsyento, pondo, nangyari,...",0.018451,1.0,False
71,Benigno S. Aquino III,"July 25, 2011",Second State of the Nation Address,https://www.officialgazette.gov.ph/2011/07/25/...,"Batasang Pambansa, Quezon City","Fifteenth Congress, Second Session",Senate President Juan Ponce Enrile; Speaker F...,6217,0.047288,-1.0000,negative,negative,0.447528,0.500472,0.052000,"[wangwang, taon, pilipino, dating, wala, mas, ...",-0.052944,1.0,True
72,Benigno S. Aquino III,"July 23, 2012",Third State of the Nation Address,https://www.officialgazette.gov.ph/2012/07/23/...,"Batasang Pambansa, Quezon City","Fifteenth Congress, Third Session",Senate President Juan Ponce Enrile; Speaker F...,9515,0.031271,-1.0000,negative,positive,0.502949,0.424943,0.072107,"[taon, ninyo, noong, dati, pilipino, pagbabago...",0.078006,1.0,False
73,Benigno S. Aquino III,"July 22, 2013",Fourth State of the Nation Address,https://www.officialgazette.gov.ph/2013/07/22/...,"Batasang Pambansa, Quezon City","Sixteenth Congress, First Session",Vice President Jejomar Binay; Senate Presiden...,12890,0.063955,-1.0000,negative,positive,0.656373,0.290636,0.052991,"[noong, taon, mas, di, ninyo, pesos, pulis, pi...",0.365738,1.0,False
74,Benigno S. Aquino III,"July 28, 2014",Fifth State of the Nation Address,https://www.officialgazette.gov.ph/2014/07/28/...,"Batasang Pambansa, Quezon City","Sixteenth Congress, Second Session",Vice President Jejomar Binay; President Fidel...,10234,0.061411,-1.0000,negative,positive,0.759564,0.239055,0.001381,"[mas, taon, tiwala, noong, boss, ninyo, matapo...",0.520509,1.0,False
75,Benigno S. Aquino III,"July 27, 2015",Sixth State of the Nation Address,https://www.officialgazette.gov.ph/2015/07/27/...,"Batasang Pambansa, Quezon City","Sixteenth Congress, Third Session","Thank you, everyone. Please sit down. Before ...",14943,0.029932,-1.0000,negative,positive,0.718865,0.237461,0.043674,"[noong, di, taon, matuwid, pagbabago, mas, nin...",0.481405,1.0,False


## Run NLTK again

This time with English Aquino speeches.

In [33]:
sia = SentimentIntensityAnalyzer()

def get_scores(content):
    blob = TextBlob(content)
    sia_scores = sia.polarity_scores(content)

    return pd.Series({
        'textblob_english': blob.sentiment.polarity,
        'nltk_english': sia_scores['compound'],
    })

df['speech'] = df['speech'].fillna('')
scores = df['speech'].apply(get_scores)
df = pd.concat([df, scores], axis=1)

def label_sentiment(score):
    if score >= 0.05:
        return 'positive'
    elif score <= -0.05:
        return 'negative'
    else:
        return 'neutral'

df['sentiment_label_english'] = df['nltk_english'].apply(label_sentiment)
df

,president,date,title,link,venue,session,speech,total_words,textblob,nltk,...,pos_score,neg_score,neutral_score,top_tfidf_words,overall_sentiment_score,total_score,matched_results,textblob_english,nltk_english,sentiment_label_english
0,Manuel L. Quezon,"November 25, 1935",Message to the First Assembly on National Defense,https://www.officialgazette.gov.ph/1935/11/25/...,"Legislative Building, Manila","First National Assembly, First Session","Mr. Speaker, gentlemen of the National Assemb...",4341,0.106794,0.9999,...,0.457971,0.541995,0.000034,"[army, defense, military, defensive, training,...",-0.084024,1.0,False,0.106794,0.9999,positive
1,Manuel L. Quezon,"June 16, 1936",On the Country’s Conditions and Problems,https://www.officialgazette.gov.ph/1936/06/16/...,"Legislative Building, Manila","First National Assembly, First Session","Mr. Speaker, Gentlemen of the National Assemb...",7250,0.110458,1.0000,...,0.457538,0.538548,0.003914,"[government, railroad, national, commonwealth,...",-0.081010,1.0,False,0.110458,1.0000,positive
2,Manuel L. Quezon,"October 18, 1937","Improvement of Philippine Conditions, Philippi...",https://www.officialgazette.gov.ph/1937/10/18/...,"Legislative Building, Manila","First National Assembly, Second Session","Mr. Speaker, Gentlemen of the National Assemb...",5774,0.137388,1.0000,...,0.494213,0.505611,0.000176,"[independence, philippines, committee, governm...",-0.011398,1.0,False,0.137388,1.0000,positive
3,Manuel L. Quezon,"January 24, 1938",Revision of the System of Taxation,https://www.officialgazette.gov.ph/1938/01/24/...,"Legislative Building, Manila","First National Assembly, Third Session",Gentlemen of the National Assembly: The state...,3212,0.060580,0.9995,...,0.105746,0.806921,0.087333,"[tax, taxes, sales, income, government, paid, ...",-0.701175,1.0,False,0.060580,0.9995,positive
4,Manuel L. Quezon,"January 24, 1939",The State of the Nation and Important Economic...,https://www.officialgazette.gov.ph/1939/01/24/...,"Legislative Building, Manila","Second National Assembly, First Session",Gentlemen of the National Assembly: I take pl...,4826,0.121574,1.0000,...,0.822032,0.177949,0.000019,"[assembly, national, recommendations, universi...",0.644082,1.0,True,0.121574,1.0000,positive
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81,Rodrigo Roa Duterte,"July 26, 2021",Sixth State of the Nation Address,https://www.officialgazette.gov.ph/2021/07/26/...,"Batasang Pambansa, Quezon City","Eighteenth Congress, Third Session",Kindly sit down. By far this is the most bea...,14769,0.133088,1.0000,...,0.411494,0.588375,0.000131,"[covid, itong, pandemic, talaga, ninyo, ganoon...",-0.176881,1.0,False,0.133088,1.0000,positive
82,Ferdinand R. Marcos Jr.,"July 25, 2022",First State of the Nation Address,https://www.officialgazette.gov.ph/2021/07/26/...,"Batasang Pambansa, Quezon City","Nineteenth Congress, First Session",Vice President Sara Zimmerman Duterte; Form...,7990,0.119099,1.0000,...,0.997522,0.002124,0.000353,"[pandemic, seeks, covid, railway, department, ...",0.995398,1.0,True,0.119099,1.0000,positive
83,Ferdinand R. Marcos Jr.,"July 24, 2023",Second State of the Nation Address,https://www.officialgazette.gov.ph/sf0clA,"Batasang Pambansa, Quezon City","Nineteenth Congress, Second Session",Thank you. Thank you very much. Allow me to g...,8227,0.139571,1.0000,...,0.999729,0.000248,0.000023,"[percent, mahigit, libong, government, buong, ...",0.999481,1.0,True,0.139571,1.0000,positive
84,Ferdinand R. Marcos Jr.,"July 22, 2024",Third State of the Nation Address,https://www.officialgazette.gov.ph/YdHHeA,"Batasang Pambansa, Quezon City","Nineteenth Congress, Third Session",Thank you. Allow me to greet the former Presi...,9177,0.139383,1.0000,...,0.934042,0.037473,0.028485,"[libong, mahigit, bansa, taon, barmm, buong, t...",0.896570,1.0,True,0.139383,1.0000,positive


In [34]:
df.sentiment_label_english.value_counts()
## Aquino's speeches became positive

positive    84
negative     2
Name: sentiment_label_english, dtype: int64

## Match with RoBERTa in English

In [35]:
df['matched_results_with_aquino_english'] = df['sentiment_label_english'] == df['roberta_label']
df

,president,date,title,link,venue,session,speech,total_words,textblob,nltk,...,neg_score,neutral_score,top_tfidf_words,overall_sentiment_score,total_score,matched_results,textblob_english,nltk_english,sentiment_label_english,matched_results_with_aquino_english
0,Manuel L. Quezon,"November 25, 1935",Message to the First Assembly on National Defense,https://www.officialgazette.gov.ph/1935/11/25/...,"Legislative Building, Manila","First National Assembly, First Session","Mr. Speaker, gentlemen of the National Assemb...",4341,0.106794,0.9999,...,0.541995,0.000034,"[army, defense, military, defensive, training,...",-0.084024,1.0,False,0.106794,0.9999,positive,False
1,Manuel L. Quezon,"June 16, 1936",On the Country’s Conditions and Problems,https://www.officialgazette.gov.ph/1936/06/16/...,"Legislative Building, Manila","First National Assembly, First Session","Mr. Speaker, Gentlemen of the National Assemb...",7250,0.110458,1.0000,...,0.538548,0.003914,"[government, railroad, national, commonwealth,...",-0.081010,1.0,False,0.110458,1.0000,positive,False
2,Manuel L. Quezon,"October 18, 1937","Improvement of Philippine Conditions, Philippi...",https://www.officialgazette.gov.ph/1937/10/18/...,"Legislative Building, Manila","First National Assembly, Second Session","Mr. Speaker, Gentlemen of the National Assemb...",5774,0.137388,1.0000,...,0.505611,0.000176,"[independence, philippines, committee, governm...",-0.011398,1.0,False,0.137388,1.0000,positive,False
3,Manuel L. Quezon,"January 24, 1938",Revision of the System of Taxation,https://www.officialgazette.gov.ph/1938/01/24/...,"Legislative Building, Manila","First National Assembly, Third Session",Gentlemen of the National Assembly: The state...,3212,0.060580,0.9995,...,0.806921,0.087333,"[tax, taxes, sales, income, government, paid, ...",-0.701175,1.0,False,0.060580,0.9995,positive,False
4,Manuel L. Quezon,"January 24, 1939",The State of the Nation and Important Economic...,https://www.officialgazette.gov.ph/1939/01/24/...,"Legislative Building, Manila","Second National Assembly, First Session",Gentlemen of the National Assembly: I take pl...,4826,0.121574,1.0000,...,0.177949,0.000019,"[assembly, national, recommendations, universi...",0.644082,1.0,True,0.121574,1.0000,positive,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81,Rodrigo Roa Duterte,"July 26, 2021",Sixth State of the Nation Address,https://www.officialgazette.gov.ph/2021/07/26/...,"Batasang Pambansa, Quezon City","Eighteenth Congress, Third Session",Kindly sit down. By far this is the most bea...,14769,0.133088,1.0000,...,0.588375,0.000131,"[covid, itong, pandemic, talaga, ninyo, ganoon...",-0.176881,1.0,False,0.133088,1.0000,positive,False
82,Ferdinand R. Marcos Jr.,"July 25, 2022",First State of the Nation Address,https://www.officialgazette.gov.ph/2021/07/26/...,"Batasang Pambansa, Quezon City","Nineteenth Congress, First Session",Vice President Sara Zimmerman Duterte; Form...,7990,0.119099,1.0000,...,0.002124,0.000353,"[pandemic, seeks, covid, railway, department, ...",0.995398,1.0,True,0.119099,1.0000,positive,True
83,Ferdinand R. Marcos Jr.,"July 24, 2023",Second State of the Nation Address,https://www.officialgazette.gov.ph/sf0clA,"Batasang Pambansa, Quezon City","Nineteenth Congress, Second Session",Thank you. Thank you very much. Allow me to g...,8227,0.139571,1.0000,...,0.000248,0.000023,"[percent, mahigit, libong, government, buong, ...",0.999481,1.0,True,0.139571,1.0000,positive,True
84,Ferdinand R. Marcos Jr.,"July 22, 2024",Third State of the Nation Address,https://www.officialgazette.gov.ph/YdHHeA,"Batasang Pambansa, Quezon City","Nineteenth Congress, Third Session",Thank you. Allow me to greet the former Presi...,9177,0.139383,1.0000,...,0.037473,0.028485,"[libong, mahigit, bansa, taon, barmm, buong, t...",0.896570,1.0,True,0.139383,1.0000,positive,True


In [36]:
df.matched_results_with_aquino_english.value_counts()

True     71
False    15
Name: matched_results_with_aquino_english, dtype: int64

In [37]:
df[df.matched_results_with_aquino_english==False]

,president,date,title,link,venue,session,speech,total_words,textblob,nltk,...,neg_score,neutral_score,top_tfidf_words,overall_sentiment_score,total_score,matched_results,textblob_english,nltk_english,sentiment_label_english,matched_results_with_aquino_english
0,Manuel L. Quezon,"November 25, 1935",Message to the First Assembly on National Defense,https://www.officialgazette.gov.ph/1935/11/25/...,"Legislative Building, Manila","First National Assembly, First Session","Mr. Speaker, gentlemen of the National Assemb...",4341,0.106794,0.9999,...,0.541995,0.000034,"[army, defense, military, defensive, training,...",-0.084024,1.0,False,0.106794,0.9999,positive,False
1,Manuel L. Quezon,"June 16, 1936",On the Country’s Conditions and Problems,https://www.officialgazette.gov.ph/1936/06/16/...,"Legislative Building, Manila","First National Assembly, First Session","Mr. Speaker, Gentlemen of the National Assemb...",7250,0.110458,1.0000,...,0.538548,0.003914,"[government, railroad, national, commonwealth,...",-0.081010,1.0,False,0.110458,1.0000,positive,False
2,Manuel L. Quezon,"October 18, 1937","Improvement of Philippine Conditions, Philippi...",https://www.officialgazette.gov.ph/1937/10/18/...,"Legislative Building, Manila","First National Assembly, Second Session","Mr. Speaker, Gentlemen of the National Assemb...",5774,0.137388,1.0000,...,0.505611,0.000176,"[independence, philippines, committee, governm...",-0.011398,1.0,False,0.137388,1.0000,positive,False
3,Manuel L. Quezon,"January 24, 1938",Revision of the System of Taxation,https://www.officialgazette.gov.ph/1938/01/24/...,"Legislative Building, Manila","First National Assembly, Third Session",Gentlemen of the National Assembly: The state...,3212,0.060580,0.9995,...,0.806921,0.087333,"[tax, taxes, sales, income, government, paid, ...",-0.701175,1.0,False,0.060580,0.9995,positive,False
5,Manuel L. Quezon,"January 22, 1940",The State of the Nation,https://www.officialgazette.gov.ph/1940/01/22/...,"Legislative Building, Manila","Second National Assembly, Second Session",Gentlemen of the National Assembly: You are c...,5838,0.100285,0.9999,...,0.529245,0.000075,"[government, national, assembly, owners, tobac...",-0.058566,1.0,False,0.100285,0.9999,positive,False
35,Ferdinand E. Marcos,"September 21, 1974",The Barangay and the Imperative of National Unity,https://www.officialgazette.gov.ph/1974/09/21/...,"Maharlika Hall, Malacañan Palace",NaN,“The Barangay and the Imperative of National ...,9136,0.093389,0.9998,...,0.530274,0.000142,"[barangays, samahang, barangay, nayon, addicti...",-0.060690,1.0,False,0.093389,0.9998,positive,False
36,Ferdinand E. Marcos,"September 19, 1975",The President’s Report to the Nation,https://www.officialgazette.gov.ph/1975/09/19/...,Quirino Grandstand,NaN,“The President’s Report to the Nation” Itinak...,6099,0.069883,0.9998,...,0.558754,0.000031,"[resignation, government, accept, society, res...",-0.117538,1.0,False,0.069883,0.9998,positive,False
43,Ferdinand E. Marcos,"July 26, 1982",State-of-the-Nation Address,https://www.officialgazette.gov.ph/1982/07/26/...,"Batasang Pambansa, Quezon City",Interim Batasang Pambansa,The President; Mr. Speaker; distinguished col...,5065,0.072171,0.9995,...,0.762507,0.018321,"[batasang, pambansa, constitution, government,...",-0.543334,1.0,False,0.072171,0.9995,positive,False
51,Corazon C. Aquino,"July 22, 1991",The State of the Nation,https://www.officialgazette.gov.ph/1991/07/22/...,"Batasang Pambansa, Quezon City","Eighth Congress, Fifth Session",Senate President Jovito Salonga; Speaker Ramo...,4001,0.127536,0.9999,...,0.605711,0.000025,"[people, elections, government, candle, democr...",-0.211447,1.0,False,0.127536,0.9999,positive,False
58,Joseph Ejercito Estrada,"July 27, 1998",The State of the Nation,https://www.officialgazette.gov.ph/1998/07/27/...,"Batasang Pambansa, Quezon City","Eleventh Congress, First Session",Ang Pangalawang Pangulo ng Republika ng Pilip...,3981,0.065895,0.9904,...,0.791217,0.000018,"[percent

## Remove speeches so we can check in Excel

In [38]:
df=df.drop(['speech'],axis=1)
df

,president,date,title,link,venue,session,total_words,textblob,nltk,sentiment_label,...,neg_score,neutral_score,top_tfidf_words,overall_sentiment_score,total_score,matched_results,textblob_english,nltk_english,sentiment_label_english,matched_results_with_aquino_english
0,Manuel L. Quezon,"November 25, 1935",Message to the First Assembly on National Defense,https://www.officialgazette.gov.ph/1935/11/25/...,"Legislative Building, Manila","First National Assembly, First Session",4341,0.106794,0.9999,positive,...,0.541995,0.000034,"[army, defense, military, defensive, training,...",-0.084024,1.0,False,0.106794,0.9999,positive,False
1,Manuel L. Quezon,"June 16, 1936",On the Country’s Conditions and Problems,https://www.officialgazette.gov.ph/1936/06/16/...,"Legislative Building, Manila","First National Assembly, First Session",7250,0.110458,1.0000,positive,...,0.538548,0.003914,"[government, railroad, national, commonwealth,...",-0.081010,1.0,False,0.110458,1.0000,positive,False
2,Manuel L. Quezon,"October 18, 1937","Improvement of Philippine Conditions, Philippi...",https://www.officialgazette.gov.ph/1937/10/18/...,"Legislative Building, Manila","First National Assembly, Second Session",5774,0.137388,1.0000,positive,...,0.505611,0.000176,"[independence, philippines, committee, governm...",-0.011398,1.0,False,0.137388,1.0000,positive,False
3,Manuel L. Quezon,"January 24, 1938",Revision of the System of Taxation,https://www.officialgazette.gov.ph/1938/01/24/...,"Legislative Building, Manila","First National Assembly, Third Session",3212,0.060580,0.9995,positive,...,0.806921,0.087333,"[tax, taxes, sales, income, government, paid, ...",-0.701175,1.0,False,0.060580,0.9995,positive,False
4,Manuel L. Quezon,"January 24, 1939",The State of the Nation and Important Economic...,https://www.officialgazette.gov.ph/1939/01/24/...,"Legislative Building, Manila","Second National Assembly, First Session",4826,0.121574,1.0000,positive,...,0.177949,0.000019,"[assembly, national, recommendations, universi...",0.644082,1.0,True,0.121574,1.0000,positive,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81,Rodrigo Roa Duterte,"July 26, 2021",Sixth State of the Nation Address,https://www.officialgazette.gov.ph/2021/07/26/...,"Batasang Pambansa, Quezon City","Eighteenth Congress, Third Session",14769,0.133088,1.0000,positive,...,0.588375,0.000131,"[covid, itong, pandemic, talaga, ninyo, ganoon...",-0.176881,1.0,False,0.133088,1.0000,positive,False
82,Ferdinand R. Marcos Jr.,"July 25, 2022",First State of the Nation Address,https://www.officialgazette.gov.ph/2021/07/26/...,"Batasang Pambansa, Quezon City","Nineteenth Congress, First Session",7990,0.119099,1.0000,positive,...,0.002124,0.000353,"[pandemic, seeks, covid, railway, department, ...",0.995398,1.0,True,0.119099,1.0000,positive,True
83,Ferdinand R. Marcos Jr.,"July 24, 2023",Second State of the Nation Address,https://www.officialgazette.gov.ph/sf0clA,"Batasang Pambansa, Quezon City","Nineteenth Congress, Second Session",8227,0.139571,1.0000,positive,...,0.000248,0.000023,"[percent, mahigit, libong, government, buong, ...",0.999481,1.0,True,0.139571,1.0000,positive,True
84,Ferdinand R. Marcos Jr.,"July 22, 2024",Third State of the Nation Address,https://www.officialgazette.gov.ph/YdHHeA,"Batasang Pambansa, Quezon City","Nineteenth Congress, Third Session",9177,0.139383,1.0000,positive,...,0.037473,0.028485,"[libong, mahigit, bansa, taon, barmm, buong, t...",0.896570,1.0,True,0.139383,1.0000,positive,True


In [42]:
# df.to_csv('analysis.csv', index=False)

In [40]:
# df.to_csv('sona_with_analysis.csv', index=False)

## Robustness check

In [41]:
# import shap

# # Wrap your pipeline for SHAP
# explainer = shap.Explainer(sentiment_pipeline)
# shap_values = explainer([df['speech'].iloc[1][9000:10000]])

# # Visualize word-level contributions
# shap.plots.text(shap_values[0])